# 1. Environment Check

In [1]:
import sqlalchemy
import pandas as pd

print("SQLAlchemy version:", sqlalchemy.__version__)
print("Pandas version:", pd.__version__)

SQLAlchemy version: 2.0.52
Pandas version: 3.0.5


# 2. PostgreSQL Connection

In [2]:
from getpass import getpass

password = getpass("Enter your PostgreSQL password: ")

Enter your PostgreSQL password:  ········


In [3]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

connection_url = URL.create(
    drivername="postgresql+psycopg",
    username="postgres",
    password=password,
    host="localhost",
    port=5432,
    database="quick_commerce_bi"
)

engine = create_engine(connection_url)

print("Database engine created successfully.")

Database engine created successfully.


In [4]:
from sqlalchemy import text

with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print("Database connection successful!")
    print("Test result:", result.scalar())

Database connection successful!
Test result: 1


# 3. SwiftBasket Schema Inspection

In [17]:
import pandas as pd

query = """
SELECT *
FROM swiftbasket.products
LIMIT 5;
"""

products_df = pd.read_sql(query, engine)

products_df

,product_id,sku,product_name,brand,brand_tier,category,sub_category,item,type,variant,...,selling_price,cost_price,discount_percentage,shelf_life_days,storage_type,gst_percentage,is_perishable,product_status,average_rating,launch_status
0,P000001,MCC-FRI-VEG-7500,McCain Fries & Bites Cheese Balls Veg Spicy 75...,McCain,Premium,Frozen Foods,Frozen Snacks,Fries & Bites,Cheese Balls,Veg,...,305.51,218.64,8.8,180,Frozen,18,True,Low Stock,4.7,Regular
1,P000002,BIN-CHI-ROA-900,Bingo Corn Chips Roasted Cream & Onion 90.0 g,Bingo,Budget,Snacks,Munchies,Chips,Corn Chips,Roasted,...,40.62,28.39,17.1,180,Ambient,12,False,Available,4.2,Regular
2,P000003,FOR-CHA-SEL-150,Fortune Chakki Atta Maida Select 15.0 kg,Fortune,Popular,Staples & Grocery,Atta,Chakki Atta,Maida,Select,...,817.49,602.20,4.4,90,Ambient,0,False,Available,4.2,Regular
3,P000004,FAS-BAS-LEA-30,Fastrack Basic Wear Socks Leather 3.0 pack,Fastrack,Popular,Fashion & Lifestyle,Apparel & Accessories,Basic Wear,Socks,Leather,...,292.19,231.43,13.8,3650,Ambient,12,False,Available,4.2,Seasonal
4,P000005,GAR-FAC-HYD-300,Garnier Face Serum Liquid Serum Hydrating Niac...,Garnier,Budget,Beauty & Personal Care,Skin Care,Face Serum,Liquid Serum,Hydrating,...,414.83,208.78,17.9,730,Ambient,18,False,Available,4.1,Regular


In [6]:
print("Rows retrieved:", len(products_df))
print("\nColumns:")
print(products_df.columns.tolist())

Rows retrieved: 5

Columns:
['product_id', 'sku', 'product_name', 'brand', 'brand_tier', 'category', 'sub_category', 'item', 'type', 'variant', 'flavour', 'unit', 'size', 'mrp', 'selling_price', 'cost_price', 'discount_percentage', 'shelf_life_days', 'storage_type', 'gst_percentage', 'is_perishable', 'product_status', 'average_rating', 'launch_status']


In [7]:
from sqlalchemy import text

query = """
SELECT
    table_name,
    column_name,
    data_type
FROM information_schema.columns
WHERE table_schema = 'swiftbasket'
ORDER BY table_name, ordinal_position;
"""

schema_df = pd.read_sql(query, engine)

schema_df

,table_name,column_name,data_type
0,customers,customer_id,character varying
1,customers,customer_name,character varying
2,customers,gender,character varying
3,customers,age,smallint
4,customers,age_group,character varying
...,...,...,...
105,riders,rider_name,character varying
106,riders,store_id,character varying
107,riders,rider_status,character varying
108,riders,average_rating,numeric


In [8]:
print("Tables found:")

for table in schema_df["table_name"].unique():
    print("-", table)

Tables found:
- customers
- dark_stores
- inventory
- order_details
- orders
- products
- returns
- riders


In [9]:
import pandas as pd
from sqlalchemy import text

schema_query = text(
    """
    SELECT 
        table_name,
        column_name,
        data_type,
        ordinal_position
    FROM information_schema.columns
    WHERE table_schema = 'swiftbasket'
      AND table_name IN ('products', 'orders', 'order_details', 'returns')
    ORDER BY table_name, ordinal_position;
    """
)

with engine.connect() as connection:
    df_schema = pd.read_sql(schema_query, connection)

df_schema

,table_name,column_name,data_type,ordinal_position
0,order_details,order_detail_id,character varying,1
1,order_details,order_id,character varying,2
2,order_details,product_id,character varying,3
3,order_details,quantity,integer,4
4,order_details,selling_price,numeric,5
5,order_details,item_discount,numeric,6
6,order_details,final_item_price,numeric,7
7,orders,order_id,character varying,1
8,orders,customer_id,character varying,2
9,orders,store_id,character varying,3


In [10]:
for table_name, group in df_schema.groupby("table_name", sort=True):
    print(f"{table_name.upper()}\n")
    for col in group["column_name"]:
        print(f"  - {col}")
    print("\n" + "-" * 30 + "\n")

ORDER_DETAILS

  - order_detail_id
  - order_id
  - product_id
  - quantity
  - selling_price
  - item_discount
  - final_item_price

------------------------------

ORDERS

  - order_id
  - customer_id
  - store_id
  - rider_id
  - order_timestamp
  - delivery_slot
  - payment_method
  - payment_status
  - order_status
  - cancellation_reason
  - delivery_fee
  - coupon_discount
  - estimated_delivery_time
  - actual_delivery_time
  - delivery_distance_km
  - total_order_value
  - weather_condition
  - peak_hour
  - festival_flag
  - order_source
  - customer_delivery_rating

------------------------------

PRODUCTS

  - product_id
  - sku
  - product_name
  - brand
  - brand_tier
  - category
  - sub_category
  - item
  - type
  - variant
  - flavour
  - unit
  - size
  - mrp
  - selling_price
  - cost_price
  - discount_percentage
  - shelf_life_days
  - storage_type
  - gst_percentage
  - is_perishable
  - product_status
  - average_rating
  - launch_status

-----------------------

# 4. Data Profiling

In [11]:
import pandas as pd
from sqlalchemy import text
from IPython.display import display

mvp_tables = ['products', 'orders', 'order_details', 'returns']

with engine.connect() as conn:
    # 1. Exact Row Counts
    print("--- ROW COUNTS ---")
    for table in mvp_tables:
        count_query = text(f"SELECT COUNT(*) FROM swiftbasket.{table};")
        row_count = conn.execute(count_query).scalar()
        print(f"{table}: {row_count} rows")
    
    print("\n" + "="*50 + "\n")
    
    # 2. Display 5 Representative Rows
    print("--- DATA SAMPLES (First 5 Rows) ---")
    for table in mvp_tables:
        print(f"\nTable: {table.upper()}")
        sample_query = text(f"SELECT * FROM swiftbasket.{table} LIMIT 5;")
        df_sample = pd.read_sql(sample_query, conn)
        display(df_sample)

--- ROW COUNTS ---
products: 2314 rows
orders: 300000 rows
order_details: 873055 rows
returns: 28866 rows


--- DATA SAMPLES (First 5 Rows) ---

Table: PRODUCTS


,product_id,sku,product_name,brand,brand_tier,category,sub_category,item,type,variant,...,selling_price,cost_price,discount_percentage,shelf_life_days,storage_type,gst_percentage,is_perishable,product_status,average_rating,launch_status
0,P000001,MCC-FRI-VEG-7500,McCain Fries & Bites Cheese Balls Veg Spicy 75...,McCain,Premium,Frozen Foods,Frozen Snacks,Fries & Bites,Cheese Balls,Veg,...,305.51,218.64,8.8,180,Frozen,18,True,Low Stock,4.7,Regular
1,P000002,BIN-CHI-ROA-900,Bingo Corn Chips Roasted Cream & Onion 90.0 g,Bingo,Budget,Snacks,Munchies,Chips,Corn Chips,Roasted,...,40.62,28.39,17.1,180,Ambient,12,False,Available,4.2,Regular
2,P000003,FOR-CHA-SEL-150,Fortune Chakki Atta Maida Select 15.0 kg,Fortune,Popular,Staples & Grocery,Atta,Chakki Atta,Maida,Select,...,817.49,602.20,4.4,90,Ambient,0,False,Available,4.2,Regular
3,P000004,FAS-BAS-LEA-30,Fastrack Basic Wear Socks Leather 3.0 pack,Fastrack,Popular,Fashion & Lifestyle,Apparel & Accessories,Basic Wear,Socks,Leather,...,292.19,231.43,13.8,3650,Ambient,12,False,Available,4.2,Seasonal
4,P000005,GAR-FAC-HYD-300,Garnier Face Serum Liquid Serum Hydrating Niac...,Garnier,Budget,Beauty & Personal Care,Skin Care,Face Serum,Liquid Serum,Hydrating,...,414.83,208.78,17.9,730,Ambient,18,False,Available,4.1,Regular



Table: ORDERS


,order_id,customer_id,store_id,rider_id,order_timestamp,delivery_slot,payment_method,payment_status,order_status,cancellation_reason,...,coupon_discount,estimated_delivery_time,actual_delivery_time,delivery_distance_km,total_order_value,weather_condition,peak_hour,festival_flag,order_source,customer_delivery_rating
0,ORD0000001,C003135,DS003,RID0122,2025-04-01 06:04:11,06:00 AM - 07:00 AM,UPI,Paid,Delivered,None,...,0.0,13,15,1.17,525.68,Clear,False,False,Android,4.0
1,ORD0000002,C009304,DS006,RID0214,2025-04-01 06:07:34,06:00 AM - 07:00 AM,Wallet,Paid,Delivered,None,...,0.0,12,9,0.62,311.61,Clear,False,False,Android,5.0
2,ORD0000003,C013322,DS009,RID0309,2025-04-01 06:16:36,06:00 AM - 07:00 AM,UPI,Paid,Delivered,None,...,0.0,26,29,6.40,243.52,Clear,False,False,Android,5.0
3,ORD0000004,C006007,DS008,RID0289,2025-04-01 06:18:41,06:00 AM - 07:00 AM,Credit Card,Paid,Delivered,None,...,0.0,19,23,1.76,376.11,Clear,False,True,Android,4.0
4,ORD0000005,C014321,DS007,RID0255,2025-04-01 06:19:00,06:00 AM - 07:00 AM,UPI,Paid,Delivered,None,...,0.0,14,17,1.79,232.97,Clear,False,False,Android,5.0



Table: ORDER_DETAILS


,order_detail_id,order_id,product_id,quantity,selling_price,item_discount,final_item_price
0,OD00000001,ORD0000001,P000852,1,70.27,18.58,51.69
1,OD00000002,ORD0000001,P002042,1,150.44,39.79,110.65
2,OD00000003,ORD0000001,P000357,1,69.45,18.37,51.08
3,OD00000004,ORD0000001,P001895,1,102.57,27.13,75.44
4,OD00000005,ORD0000001,P000572,1,114.51,30.28,84.23



Table: RETURNS


,return_id,order_id,product_id,return_date,return_reason,refund_amount,return_status
0,RET0000001,ORD0000003,P000954,2025-04-01,Damaged in Transit,79.15,Approved
1,RET0000002,ORD0000018,P001227,2025-04-01,Expired Product,31.21,Approved
2,RET0000003,ORD0000028,P002283,2025-04-02,Quality/Freshness Issue,0.00,Rejected
3,RET0000004,ORD0000038,P001433,2025-04-01,Wrong Item Delivered,199.97,Approved
4,RET0000005,ORD0000038,P000701,2025-04-01,Wrong Item Delivered,117.63,Approved


In [12]:
with engine.connect() as conn:
    print("--- DISTINCT ID COUNTS ---")
    
    # 3. Distinct product_id in products
    prod_query = text("SELECT COUNT(DISTINCT product_id) FROM swiftbasket.products;")
    print(f"Distinct products in 'products': {conn.execute(prod_query).scalar()}")
    
    # 4. Distinct order_id in orders
    ord_query = text("SELECT COUNT(DISTINCT order_id) FROM swiftbasket.orders;")
    print(f"Distinct orders in 'orders': {conn.execute(ord_query).scalar()}")
    
    # 5. Distinct order_id in order_details
    ord_det_query = text("SELECT COUNT(DISTINCT order_id) FROM swiftbasket.order_details;")
    print(f"Distinct orders in 'order_details': {conn.execute(ord_det_query).scalar()}")
    
    # 6. Distinct order_id in returns
    ret_ord_query = text("SELECT COUNT(DISTINCT order_id) FROM swiftbasket.returns;")
    print(f"Distinct orders in 'returns': {conn.execute(ret_ord_query).scalar()}")

--- DISTINCT ID COUNTS ---
Distinct products in 'products': 2314
Distinct orders in 'orders': 300000
Distinct orders in 'order_details': 300000
Distinct orders in 'returns': 27385


In [13]:
from sqlalchemy import text

with engine.connect() as conn:
    print("--- OPTIMIZED REFERENTIAL INTEGRITY CHECKS ---")
    print("(A result of 0 means all records match perfectly)\n")
    
    # 1. Check if order_details.order_id matches orders.order_id
    check_1 = text("""
        SELECT COUNT(*) FROM swiftbasket.order_details od
        WHERE NOT EXISTS (
            SELECT 1 FROM swiftbasket.orders o 
            WHERE o.order_id = od.order_id
        );
    """)
    print(f"Orphan order_ids in order_details: {conn.execute(check_1).scalar()}")
    
    # 2. Check if order_details.product_id matches products.product_id
    check_2 = text("""
        SELECT COUNT(*) FROM swiftbasket.order_details od
        WHERE NOT EXISTS (
            SELECT 1 FROM swiftbasket.products p 
            WHERE p.product_id = od.product_id
        );
    """)
    print(f"Orphan product_ids in order_details: {conn.execute(check_2).scalar()}")
    
    # 3. Check if returns.order_id matches orders.order_id
    check_3 = text("""
        SELECT COUNT(*) FROM swiftbasket.returns r
        WHERE NOT EXISTS (
            SELECT 1 FROM swiftbasket.orders o 
            WHERE o.order_id = r.order_id
        );
    """)
    print(f"Orphan order_ids in returns: {conn.execute(check_3).scalar()}")
    
    # 4. Check if returns.product_id matches products.product_id
    check_4 = text("""
        SELECT COUNT(*) FROM swiftbasket.returns r
        WHERE NOT EXISTS (
            SELECT 1 FROM swiftbasket.products p 
            WHERE p.product_id = r.product_id
        );
    """)
    print(f"Orphan product_ids in returns: {conn.execute(check_4).scalar()}")

--- OPTIMIZED REFERENTIAL INTEGRITY CHECKS ---
(A result of 0 means all records match perfectly)

Orphan order_ids in order_details: 0
Orphan product_ids in order_details: 0
Orphan order_ids in returns: 0
Orphan product_ids in returns: 0


# 5. Order Distribution Analysis

In [14]:
import pandas as pd
from sqlalchemy import text
from IPython.display import display

with engine.connect() as conn:
    # 1. Distinct products in order_details
    prod_count_query = text("SELECT COUNT(DISTINCT product_id) FROM swiftbasket.order_details;")
    distinct_products = conn.execute(prod_count_query).scalar()
    
    # 2. Aggregation query for distribution metrics
    agg_query = text("""
        WITH OrderItemCounts AS (
            SELECT order_id, COUNT(product_id) AS item_count
            FROM swiftbasket.order_details
            GROUP BY order_id
        )
        SELECT 
            MIN(item_count) AS min_items,
            MAX(item_count) AS max_items,
            ROUND(AVG(item_count), 2) AS avg_items,
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY item_count) AS median_items,
            COUNT(*) FILTER (WHERE item_count = 1) AS orders_1_item,
            COUNT(*) FILTER (WHERE item_count BETWEEN 2 AND 3) AS orders_2_3_items,
            COUNT(*) FILTER (WHERE item_count BETWEEN 4 AND 5) AS orders_4_5_items,
            COUNT(*) FILTER (WHERE item_count > 5) AS orders_gt_5_items
        FROM OrderItemCounts;
    """)
    
    result = conn.execute(agg_query).fetchone()
    min_items, max_items, avg_items, median_items, bin_1, bin_2_3, bin_4_5, bin_gt_5 = result
    
    # Print summary metrics
    print("--- ORDER DISTRIBUTION SUMMARY ---")
    print(f"Distinct products in order_details: {distinct_products}")
    print(f"Minimum items per order: {min_items}")
    print(f"Maximum items per order: {max_items}")
    print(f"Average items per order: {avg_items}")
    print(f"Median items per order: {median_items}")
    print("\n" + "="*40 + "\n")
    
    # Display distribution DataFrame
    print("--- ORDER ITEM FREQUENCY BUCKETS ---")
    dist_data = {
        "Basket Size": ["1 item", "2–3 items", "4–5 items", "> 5 items"],
        "Number of Orders": [bin_1, bin_2_3, bin_4_5, bin_gt_5]
    }
    
    df_distribution = pd.DataFrame(dist_data)
    display(df_distribution)

--- ORDER DISTRIBUTION SUMMARY ---
Distinct products in order_details: 2304
Minimum items per order: 1
Maximum items per order: 24
Average items per order: 2.91
Median items per order: 2.0


--- ORDER ITEM FREQUENCY BUCKETS ---


,Basket Size,Number of Orders
0,1 item,64970
1,2–3 items,148534
2,4–5 items,60936
3,> 5 items,25560


In [18]:
from sqlalchemy import text

# Helper function to handle NULLs cleanly
def safe_str(val, default="N/A"):
    return str(val) if val is not None else default

with engine.connect() as conn:
    
    print("================ PRODUCT DOCUMENT ================\n")
    # 1. Retrieve One Product
    prod_query = text("""
        SELECT 
            product_id, sku, product_name, brand, brand_tier, category, sub_category, 
            unit, size, mrp, selling_price, discount_percentage, 
            storage_type, is_perishable, average_rating, product_status, launch_status
        FROM swiftbasket.products
        LIMIT 1;
    """)
    prod = conn.execute(prod_query).mappings().fetchone()
    
    # Format Product Text
    prod_text = (
        f"Product Name: {safe_str(prod['product_name'])}. "
        f"Brand: {safe_str(prod['brand'])} (Tier: {safe_str(prod['brand_tier'])}). "
        f"Category: {safe_str(prod['category'])} > {safe_str(prod['sub_category'])}. "
        f"Size/Unit: {safe_str(prod['size'])} {safe_str(prod['unit'])}. "
        f"Pricing: MRP {safe_str(prod['mrp'])}, Selling Price {safe_str(prod['selling_price'])} "
        f"({safe_str(prod['discount_percentage'])}% off). "
        f"Storage: {safe_str(prod['storage_type'])}. Perishable: {safe_str(prod['is_perishable'])}. "
        f"Average Rating: {safe_str(prod['average_rating'])}. "
        f"Status: {safe_str(prod['product_status'])} (Launch: {safe_str(prod['launch_status'])})."
    )
    
    # Format Product Metadata
    prod_metadata = {
        "doc_type": "product",
        "product_id": prod['product_id'],
        "sku": prod['sku'],
        "category": prod['category'],
        "brand": prod['brand']
    }
    
    print("--- TEXT ---")
    print(prod_text)
    print("\n--- METADATA ---")
    print(prod_metadata)
    print("\n\n")

    
    print("================ ORDER DOCUMENT ================\n")
    # 2. Retrieve One Order (Aggregating Items)
    order_query = text("""
        SELECT 
            o.order_id, o.customer_id, o.order_timestamp, o.order_status, 
            o.payment_method, o.payment_status, o.total_order_value, 
            o.delivery_fee, o.coupon_discount, o.cancellation_reason,
            STRING_AGG(
                p.product_name || ' (Brand: ' || COALESCE(p.brand, 'N/A') || 
                ', Qty: ' || od.quantity || ', Final Item Price: ' || od.final_item_price || ')', 
                E'\\n- '
            ) AS items_list
        FROM swiftbasket.orders o
        JOIN swiftbasket.order_details od ON o.order_id = od.order_id
        JOIN swiftbasket.products p ON od.product_id = p.product_id
        GROUP BY 
            o.order_id, o.customer_id, o.order_timestamp, o.order_status, 
            o.payment_method, o.payment_status, o.total_order_value, 
            o.delivery_fee, o.coupon_discount, o.cancellation_reason
        LIMIT 1;
    """)
    order = conn.execute(order_query).mappings().fetchone()
    
    # Format Order Text
    cancel_str = f"\nCancellation Reason: {order['cancellation_reason']}" if order['cancellation_reason'] else ""
    
    order_text = (
        f"Order placed on: {safe_str(order['order_timestamp'])}.\n"
        f"Order Status: {safe_str(order['order_status'])}.{cancel_str}\n"
        f"Payment: {safe_str(order['payment_method'])} ({safe_str(order['payment_status'])}).\n"
        f"Financials: Total Order Value {safe_str(order['total_order_value'])}, "
        f"Delivery Fee {safe_str(order['delivery_fee'])}, "
        f"Coupon Discount {safe_str(order['coupon_discount'])}.\n\n"
        f"Items purchased:\n- {safe_str(order['items_list'])}"
    )
    
    # Format Order Metadata
    order_metadata = {
        "doc_type": "order",
        "order_id": order['order_id'],
        "customer_id": order['customer_id'],
        "order_status": order['order_status']
    }
    
    print("--- TEXT ---")
    print(order_text)
    print("\n--- METADATA ---")
    print(order_metadata)
    print("\n\n")


    print("================ RETURN DOCUMENT ================\n")
    # 3. Retrieve One Return
    return_query = text("""
        SELECT 
            r.return_id, r.order_id, r.product_id, r.return_date, 
            r.return_reason, r.return_status, r.refund_amount,
            p.product_name, p.brand, p.category,
            o.order_timestamp, o.order_status
        FROM swiftbasket.returns r
        JOIN swiftbasket.products p ON r.product_id = p.product_id
        JOIN swiftbasket.orders o ON r.order_id = o.order_id
        LIMIT 1;
    """)
    ret = conn.execute(return_query).mappings().fetchone()
    
    # Format Return Text
    ret_text = (
        f"Return filed on: {safe_str(ret['return_date'])}.\n"
        f"Product Returned: {safe_str(ret['product_name'])} "
        f"(Brand: {safe_str(ret['brand'])}, Category: {safe_str(ret['category'])}).\n"
        f"Reason for return: {safe_str(ret['return_reason'])}.\n"
        f"Return Status: {safe_str(ret['return_status'])}.\n"
        f"Refund Amount: {safe_str(ret['refund_amount'])}.\n"
        f"Original Order Context: Placed on {safe_str(ret['order_timestamp'])}, "
        f"Order Status {safe_str(ret['order_status'])}."
    )
    
    # Format Return Metadata
    ret_metadata = {
        "doc_type": "return",
        "return_id": ret['return_id'],
        "order_id": ret['order_id'],
        "product_id": ret['product_id'],
        "return_status": ret['return_status']
    }
    
    print("--- TEXT ---")
    print(ret_text)
    print("\n--- METADATA ---")
    print(ret_metadata)

================ PRODUCT DOCUMENT ================

--- TEXT ---
Product Name: McCain Fries & Bites Cheese Balls Veg Spicy 750.0 g. Brand: McCain (Tier: Premium). Category: Frozen Foods > Frozen Snacks. Size/Unit: 750.00 g. Pricing: MRP 335.00, Selling Price 305.51 (8.80% off). Storage: Frozen. Perishable: True. Average Rating: 4.7. Status: Low Stock (Launch: Regular).

--- METADATA ---
{'doc_type': 'product', 'product_id': 'P000001', 'sku': 'MCC-FRI-VEG-7500', 'category': 'Frozen Foods', 'brand': 'McCain'}



================ ORDER DOCUMENT ================

--- TEXT ---
Order placed on: 2025-04-01 06:04:11.
Order Status: Delivered.
Payment: UPI (Paid).
Financials: Total Order Value 525.68, Delivery Fee 0.00, Coupon Discount 0.00.

Items purchased:
- Haldiram's Chips Extruded Snacks Fried Magic Masala 150.0 g (Brand: Haldiram's, Qty: 1, Final Item Price: 51.69)
- ITC Master Chef Fries & Bites Nuggets Veg Masala 400.0 g (Brand: ITC Master Chef, Qty: 1, Final Item Price: 110.65)
- Eggoz

In [19]:
import json
from pathlib import Path
from sqlalchemy import text


def safe_val(val, default="N/A"):
    """Return string representation of a value or default if None/empty."""
    if val is None or str(val).strip() == "":
        return default
    return str(val)


def build_product_text(row: dict) -> str:
    """Transform a single product record into a rich, human-readable text representation."""
    # Attribute segments handling optional/variant details cleanly
    details = []
    if row.get("item"):
        details.append(f"Item: {row['item']}")
    if row.get("type"):
        details.append(f"Type: {row['type']}")
    if row.get("variant"):
        details.append(f"Variant: {row['variant']}")
    if row.get("flavour"):
        details.append(f"Flavour: {row['flavour']}")

    details_str = (
        f" Product Specifics: {', '.join(details)}." if details else ""
    )

    perishable_str = (
        "Yes"
        if row.get("is_perishable") is True
        else ("No" if row.get("is_perishable") is False else "N/A")
    )

    text_content = (
        f"Product Name: {safe_val(row.get('product_name'))}. "
        f"Brand: {safe_val(row.get('brand'))} (Tier: {safe_val(row.get('brand_tier'))}). "
        f"Category: {safe_val(row.get('category'))} > {safe_val(row.get('sub_category'))}.{details_str} "
        f"Package Size: {safe_val(row.get('size'))} {safe_val(row.get('unit'))}. "
        f"Pricing: MRP {safe_val(row.get('mrp'))}, Selling Price {safe_val(row.get('selling_price'))} "
        f"({safe_val(row.get('discount_percentage'))}% discount). "
        f"Storage Type: {safe_val(row.get('storage_type'))}. Perishable: {perishable_str}. "
        f"Shelf Life: {safe_val(row.get('shelf_life_days'))} days. "
        f"Customer Rating: {safe_val(row.get('average_rating'))} / 5.0. "
        f"Product Status: {safe_val(row.get('product_status'))} (Launch Status: {safe_val(row.get('launch_status'))})."
    )
    return text_content


def build_product_metadata(row: dict) -> dict:
    """Extract structured filter fields for vector database metadata."""
    return {
        "doc_type": "product",
        "product_id": safe_val(row.get("product_id")),
        "sku": safe_val(row.get("sku")),
        "category": safe_val(row.get("category")),
        "sub_category": safe_val(row.get("sub_category")),
        "brand": safe_val(row.get("brand")),
        "brand_tier": safe_val(row.get("brand_tier")),
        "product_status": safe_val(row.get("product_status")),
        "launch_status": safe_val(row.get("launch_status")),
        "is_perishable": (
            bool(row.get("is_perishable"))
            if row.get("is_perishable") is not None
            else False
        ),
    }


# 1. Fetch all product rows from PostgreSQL
query = text("""
    SELECT 
        product_id, sku, product_name, brand, brand_tier, category, sub_category,
        item, type, variant, flavour, unit, size, mrp, selling_price,
        discount_percentage, shelf_life_days, storage_type, is_perishable,
        product_status, average_rating, launch_status
    FROM swiftbasket.products
    ORDER BY product_id;
""")

with engine.connect() as conn:
    rows = conn.execute(query).mappings().fetchall()

total_rows_retrieved = len(rows)

# 2. Convert database rows to standardized RAG Documents
documents = []
seen_ids = set()

for row in rows:
    doc_id = str(row["product_id"]).strip()
    doc_text = build_product_text(row)
    doc_meta = build_product_metadata(row)

    # Validations per document
    if not doc_id:
        raise ValueError(f"Found record with missing product_id: {row}")
    if not doc_text or doc_text.strip() == "":
        raise ValueError(
            f"Generated empty text for product_id: {row['product_id']}"
        )
    if doc_id in seen_ids:
        raise ValueError(f"Duplicate product_id detected: {doc_id}")

    seen_ids.add(doc_id)

    documents.append({"id": doc_id, "text": doc_text, "metadata": doc_meta})

# 3. Destination setup and writing to JSONL
output_path = Path(r"E:\Python Projects\AI-Project\data\products.jsonl")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    for doc in documents:
        f.write(json.dumps(doc, ensure_ascii=False) + "\n")

# 4. Final Validations & Reporting
print("=" * 60)
print("SWIFTBASKET PRODUCT CORPUS GENERATION COMPLETED")
print("=" * 60)
print(f"Output File Path: {output_path.resolve()}")
print(f"Total Rows Retrieved from DB: {total_rows_retrieved}")
print(f"Total Documents Generated:    {len(documents)}")
print(
    f"Unique Document IDs:          {len(seen_ids)} (All IDs verified unique)"
)
print(
    f"Integrity Match:              {len(documents) == total_rows_retrieved and len(documents) == len(seen_ids)}"
)

print("\n--- FIRST DOCUMENT SAMPLE ---")
print(json.dumps(documents[0], indent=2))

print("\n--- LAST DOCUMENT SAMPLE ---")
print(json.dumps(documents[-1], indent=2))

SWIFTBASKET PRODUCT CORPUS GENERATION COMPLETED
Output File Path: E:\Python Projects\AI-Project\data\products.jsonl
Total Rows Retrieved from DB: 2314
Total Documents Generated:    2314
Unique Document IDs:          2314 (All IDs verified unique)
Integrity Match:              True

--- FIRST DOCUMENT SAMPLE ---
{
  "id": "P000001",
  "text": "Product Name: McCain Fries & Bites Cheese Balls Veg Spicy 750.0 g. Brand: McCain (Tier: Premium). Category: Frozen Foods > Frozen Snacks. Product Specifics: Item: Fries & Bites, Type: Cheese Balls, Variant: Veg, Flavour: Spicy. Package Size: 750.00 g. Pricing: MRP 335.00, Selling Price 305.51 (8.80% discount). Storage Type: Frozen. Perishable: Yes. Shelf Life: 180 days. Customer Rating: 4.7 / 5.0. Product Status: Low Stock (Launch Status: Regular).",
  "metadata": {
    "doc_type": "product",
    "product_id": "P000001",
    "sku": "MCC-FRI-VEG-7500",
    "category": "Frozen Foods",
    "sub_category": "Frozen Snacks",
    "brand": "McCain",
    "